<a href="https://colab.research.google.com/github/22f2000792/MLP-prac/blob/main/mlp_pppe1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

In [2]:
path='/content/drive/MyDrive/Dataset/mlp oppe_1/Data_preprocessing_V3.csv'

In [3]:
df=pd.read_csv(path)
df.head(3)

,Age,Experience,Projects,HoursPerWeek,EducationYears,Department,EducationLevel,City,Salary
0,37,19.0,31.0,NaN,10,IT,HighSchool,Kolkata,85543
1,23,0.0,13.0,20.0,10,Finance,PhD,Delhi,33227
2,30,0.0,28.0,NaN,17,IT,Bachelors,Bengaluru,44488


# Using the given dataset, determine which city has the lowest percentage of employees

In [4]:
df.City.value_counts()

,count
City,
Delhi,60
Mumbai,55
Bengaluru,54
Chennai,52
Kolkata,50


# Using the standard quartile approach find the number of outliers in the Salary coloum of the dataframe.

For outliers only consider values strictly greater than or strictly less than the testing conditions only.

In [5]:
Q1=df['Salary'].quantile(0.25)
Q3=df['Salary'].quantile(0.75)
IQR=Q3-Q1
lower_bound=Q1-1.5*IQR
upper_bound=Q3+1.5*IQR
outlier=df[(df['Salary']<lower_bound)|(df['Salary']>upper_bound)]

In [6]:
outlier

,Age,Experience,Projects,HoursPerWeek,EducationYears,Department,EducationLevel,City,Salary
59,45,17.0,25.0,28.0,14,Sales,PhD,Delhi,300269
92,51,17.0,40.0,51.0,21,Sales,HighSchool,NaN,312702
119,33,12.0,13.0,28.0,15,Finance,PhD,Mumbai,213409


In [7]:
len(outlier)

3

# Find the number of employees in Delhi who work more than 40 hours each week.

In [8]:
df[(df['City']=='Delhi')& (df['HoursPerWeek']>40)].shape[0]

35

In [9]:
((df.City=='Delhi')& (df.HoursPerWeek>40)).sum()

np.int64(35)

# Determine the department with the smallest variation in salary, measured using the Coefficient of Variation (CV = Standard Deviation / Mean).

In [10]:
std_=df.groupby('Department')['Salary'].std()
mean_=df.groupby('Department')['Salary'].mean()

In [11]:
std_/mean_

,Salary
Department,
Finance,0.426061
HR,0.377160
IT,0.437460
Sales,0.488568


# Perform a correlation analysis on the numerical features and the label (Salary). Which of the following features exhibits the highest positive correlation with the "Salary" label and what is the value of this correlation?

In [12]:
corr_=df.corr(numeric_only=True)
corr_['Salary'].sort_values(ascending=False)

,Salary
Salary,1.000000
Experience,0.715559
Age,0.695421
Projects,0.220419
EducationYears,0.079364
HoursPerWeek,0.019781


# Perform data imputation in the following manner:

Numerical values with median value of the feature

Categorical values with the the modal feature value

After imputation find the feature which has the lowest standard deviation.

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 295 entries, 0 to 294
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             295 non-null    int64  
 1   Experience      271 non-null    float64
 2   Projects        271 non-null    float64
 3   HoursPerWeek    271 non-null    float64
 4   EducationYears  295 non-null    int64  
 5   Department      295 non-null    object 
 6   EducationLevel  271 non-null    object 
 7   City            271 non-null    object 
 8   Salary          295 non-null    int64  
dtypes: float64(3), int64(3), object(3)
memory usage: 20.9+ KB


In [14]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer,make_column_selector
pipe_num=Pipeline([
    ('num',SimpleImputer(strategy='median')),
])
pipe_cat=Pipeline([
    ('cat',SimpleImputer(strategy='most_frequent'))
])
transformer=ColumnTransformer([
    ('pipe1',pipe_num,make_column_selector(dtype_include=np.number)),
    ('pipe2',pipe_cat,make_column_selector(dtype_exclude=np.number))
],remainder='passthrough',verbose_feature_names_out=False).set_output(transform='pandas')

In [15]:
df_proc=transformer.fit_transform(df)
df.head(3)

,Age,Experience,Projects,HoursPerWeek,EducationYears,Department,EducationLevel,City,Salary
0,37,19.0,31.0,NaN,10,IT,HighSchool,Kolkata,85543
1,23,0.0,13.0,20.0,10,Finance,PhD,Delhi,33227
2,30,0.0,28.0,NaN,17,IT,Bachelors,Bengaluru,44488


In [16]:
df_proc.describe()

,Age,Experience,Projects,HoursPerWeek,EducationYears,Salary
count,295.000000,295.000000,295.000000,295.000000,295.000000,295.000000
mean,40.054237,14.247458,22.013559,54.606780,16.342373,89699.477966
std,12.853588,10.806419,12.480181,26.620691,3.769020,38923.722676
min,18.000000,0.000000,1.000000,18.000000,10.000000,29569.000000
25%,29.000000,4.500000,12.000000,37.000000,14.000000,58636.500000
50%,39.000000,13.000000,21.000000,54.000000,16.000000,83909.000000
75%,49.500000,22.000000,32.000000,66.500000,18.000000,111276.500000
max,71.000000,42.000000,51.000000,240.000000,25.000000,312702.000000


In [17]:
df_proc.select_dtypes(include=np.number).std()

,0
Age,12.853588
Experience,10.806419
Projects,12.480181
HoursPerWeek,26.620691
EducationYears,3.769020
Salary,38923.722676


# "For this Question , use the preprocessed data obtained in the previous question"

Now perform encoding on the categorical features according to the following:

    One Hot Encoding for : City

    Ordinal Encoding for : EducationLevel

Drop the original columns after encoding them.

Drop the department column directly.

Enter the ratio of the mean of the encoded EducationLevel column to the total number of the columns in the modified dataframe. (Round to 3 decimals)

In [18]:
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
ohe_=OneHotEncoder(sparse_output=False)
ode_=OrdinalEncoder()
City_encoded=ohe_.fit_transform(df[['City']])
edu_encoded=ode_.fit_transform(df[['EducationLevel']])

In [19]:
new_df=pd.DataFrame(City_encoded,columns=ohe_.get_feature_names_out())

In [20]:
new_df['EdcationLevel']=edu_encoded
new_df.head(3)

,City_Bengaluru,City_Chennai,City_Delhi,City_Kolkata,City_Mumbai,City_nan,EdcationLevel
0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1,0.0,0.0,1.0,0.0,0.0,0.0,3.0
2,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [21]:
new_df['EdcationLevel'].mean()/len(new_df.columns)

np.float64(0.20716921454928833)

# In the original dataset create a new column NonEducationYears:

NonEducationYears = Age - Education Years

Now find the number of employees who's ratio of NonEducationYears/Age is greater than 0.7 and earn a salary > 100000.

In [22]:
df.head()

,Age,Experience,Projects,HoursPerWeek,EducationYears,Department,EducationLevel,City,Salary
0,37,19.0,31.0,NaN,10,IT,HighSchool,Kolkata,85543
1,23,0.0,13.0,20.0,10,Finance,PhD,Delhi,33227
2,30,0.0,28.0,NaN,17,IT,Bachelors,Bengaluru,44488
3,58,NaN,4.0,NaN,13,HR,Masters,NaN,107838
4,50,27.0,44.0,32.0,25,HR,HighSchool,Bengaluru,174154


In [23]:
df['NonEdu']=df['Age']-df['EducationYears']

In [24]:
df

,Age,Experience,Projects,HoursPerWeek,EducationYears,Department,EducationLevel,City,Salary,NonEdu
0,37,19.0,31.0,NaN,10,IT,HighSchool,Kolkata,85543,27
1,23,0.0,13.0,20.0,10,Finance,PhD,Delhi,33227,13
2,30,0.0,28.0,NaN,17,IT,Bachelors,Bengaluru,44488,13
3,58,NaN,4.0,NaN,13,HR,Masters,NaN,107838,45
4,50,27.0,44.0,32.0,25,HR,HighSchool,Bengaluru,174154,25
...,...,...,...,...,...,...,...,...,...,...
290,46,29.0,7.0,179.0,17,HR,Bachelors,Mumbai,104054,29
291,31,5.0,1.0,52.0,18,IT,HighSchool,Kolkata,50936,13
292,33,19.0,6.0,78.0,14,Sales,Bachelors,NaN,91228,19
293,61,22.0,NaN,67.0,17,IT,Bachelors,Mumbai,134256,44


In [25]:
#ratio=df['NonEdu']/df['Age']
#df['ratio']=ratio

In [26]:
df[((df['NonEdu']/df['Age'])>0.7)&(df['Salary']>100000)].shape[0]

48

# Part 2

In [35]:
path1='/content/drive/MyDrive/Dataset/mlp oppe_1/Model_Building_V1.csv'

In [36]:
df_raw=pd.read_csv(path1)
df_raw.head(3)

,Age,Experience,Projects,HoursPerWeek,EducationYears,Department,EducationLevel,City,Salary
0,49.0,19.0,30.0,48.0,21.171059,1,1,0,117583.304750
1,60.0,36.0,22.0,52.0,16.000000,1,0,0,149581.394501
2,23.0,0.0,29.0,52.0,21.000000,1,0,0,46209.000000


# Separate the features and the label.
We will now use this dataset for a regression task, where the Salary column will be treated as the label.

Split the data into 80% training set and 20% test set, using random_state = 0.

Keep model parameters default unless specified in the question.

Which dataset are you using for this exam?

In [37]:
feature=df_raw.drop('Salary',axis=1)
label=df_raw['Salary'].copy()

In [38]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(feature,label,test_size=0.2,random_state=0)
print(X_train.shape,y_train.shape)
print(X_test.shape,y_test.shape)

(216, 8) (216,)
(54, 8) (54,)


## Perform RFE (recursive feature elimination) on the training data.

Keep the base model as a simple LinearRegression Model with default parameters.

Have n_features_to_select as 0.7

After performing the RFE identify the feature with the highest ranking.

In [39]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
selector=RFE(LinearRegression(),n_features_to_select=0.7)
selector.fit(X_train,y_train)

RFE(estimator=LinearRegression(), n_features_to_select=0.7)

In [42]:
selector.ranking_

array([1, 1, 2, 4, 1, 1, 1, 3])

In [43]:
np.argmax(selector.ranking_)

np.int64(3)

# Fit a Lasso Model on the training dataset with the following specifications:

1.Remove the model intercept.

2.Keep warm start as true

3.Train the model for maximum 500 iterations

Compute the explained_varience_score on test set and enter its value (upto 2 decimals).

In [56]:
from sklearn.linear_model import Lasso
ls=Lasso(warm_start=True,max_iter=500,random_state=0)
ls.fit(X_train,y_train)

Lasso(max_iter=500, random_state=0, warm_start=True)

In [57]:
from sklearn.metrics import explained_variance_score
pre_test=ls.predict(X_test)
explained_variance_score(y_test,pre_test)


0.867603635720912

# Fit a Voting Regressor on the training set with the following specifications:

    Estimator 1 -> LinearRegression
    Estimator 2 -> RandomForest Regressor with 50 estimators
    Estimator 3 -> KNNRegressor with euclidian distance metric and k=4


After fitting the Voting Regressor on the training set find the average of the predicted salary in the test set. Enter upto 2 decimals.

In [83]:
from sklearn.ensemble import VotingRegressor,RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor

rd=RandomForestRegressor()
kn=KNeighborsRegressor()
vt=VotingRegressor([
    ('lr',LinearRegression()),
    ('rd',RandomForestRegressor(n_estimators=50,random_state=0)),
    ('kn',KNeighborsRegressor(n_neighbors=4,metric='euclidean'))
]).set_output(transform='pandas')
vt.fit(X_train,y_train)

VotingRegressor(estimators=[('lr', LinearRegression()),
                            ('rd',
                             RandomForestRegressor(n_estimators=50,
                                                   random_state=0)),
                            ('kn',
                             KNeighborsRegressor(metric='euclidean',
                                                 n_neighbors=4))])

In [84]:
vt_pr=vt.predict(X_test)

In [86]:
vt_pr.mean()

np.float64(79568.68472200549)

# Create a pipeline of the PCA() as transformer and Ridge as an estimator.

Use GridSearchCV for tuning the hyperparameters of the created pipeline on training dataset.

    values of n_components for PCA to be [0.9,0.95]
    ridge alpha value to be taken as : [10, 1, 0.01, 0.001]
    scoring : neg_mean_absolute_error.
    cv = 5
    n_jobs = -1 (negative one)
If we fit the pipeline on the training dataset, what will be the R2 score on the test dataset? Enter your answer correct to 2 decimal places.

In [91]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA
pipe=Pipeline([
    ('pca',PCA(random_state=0)),
    ('rg',Ridge(random_state=0))
])
param_grid={
    'pca__n_components':[0.9,0.95],
    'rg__alpha':[10, 1, 0.01, 0.001]
}
gs=GridSearchCV(estimator=pipe,param_grid=param_grid,scoring='neg_mean_absolute_error',cv=5,n_jobs=-1)
gs.fit(X_train,y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('pca', PCA(random_state=0)),
                                       ('rg', Ridge(random_state=0))]),
             n_jobs=-1,
             param_grid={'pca__n_components': [0.9, 0.95],
                         'rg__alpha': [10, 1, 0.01, 0.001]},
             scoring='neg_mean_absolute_error')

In [92]:
gs_pre=gs.predict(X_test)

In [94]:
from sklearn.metrics import r2_score
r2_score(y_test,gs_pre)

0.8729833204491915

# Create a pipeline of the PCA() as transformer and Ridge as an estimator.

Use GridSearchCV for tuning the hyperparameters of the created pipeline on training dataset.

    values of n_components for PCA to be [0.9,0.95]
    ridge alpha value to be taken as : [10, 1, 0.01, 0.001]
    scoring : neg_mean_absolute_error.
    cv = 5
    n_jobs = -1 (negative one)

(Note: Kindly ignore the warning.)
How much variance is explained by the second principle component? Enter your answer correct to two decimal places.

In [117]:
steps_=gs.best_estimator_.named_steps['pca']
steps_.explained_variance_ratio_[1]

np.float64(0.35305183517541505)